# Mellanox 对 NVDA Data Center 同比的来源定义影响

固定两个原始 SEC 对照和四个过渡同比事件；只确认来源语义，不读取 SOXX 结果。


In [ ]:
import sys,json,hashlib,re,tarfile,tempfile,subprocess,shlex
from pathlib import Path
from bs4 import BeautifulSoup
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import _settings,nas_api_call
card=root/'research/stage1_nvda_datacenter_mellanox_semantics_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('new_HTTP_GETs','market_outcome_reads','model_fits','native_accounts','new_Qlib_Recorders'))
parent=root/src['parent_numeric_card'];assert sha(parent/'result.json')==src['parent_result_sha256'] and sha(parent/'summary.json')==src['parent_summary_sha256']
pr=json.loads((parent/'result.json').read_text());events={e['accession']:e for e in pr['events']};assert len(plan['exact_definition_transition_accessions'])==4 and len(plan['source_text_controls'])==2
inventory=json.loads((root/'research/stage1_amd_nvda_sec_guidance_original_probe_20261023/item202_inventory_result.json').read_text());assert inventory['NAS_archive_sha256']==src['raw_NAS_archive_sha256'] and inventory['NAS_remote_and_scoped_restore_verified']
patterns={'pre_Q1_outlook_includes_next_quarter':r'(?i)outlook.{0,220}(?:second quarter|Q2).{0,180}Mellanox|Mellanox.{0,220}(?:outlook|second quarter)', 'Q2_closed_April_27':r'(?i)(?:closed|completed).{0,100}Mellanox.{0,80}April 27, 2020|Mellanox.{0,180}April 27, 2020', 'Q2_Data_Center_includes_Mellanox':r'(?i)Data Center revenue.{0,80}includes Mellanox|Data Center.{0,180}Mellanox contributed|Mellanox.{0,180}Data Center revenue'}
controls=[]
for i,item in enumerate(plan['source_text_controls']):
 p=root/src['original_raw_root']/src['original_raw_template'].format(accession=item['accession']);payload=p.read_bytes();assert hashlib.sha256(payload).hexdigest()==item['raw_sha256']
 full=payload.decode('utf-8','replace');docs=[m.group(1) for m in re.finditer(r'<DOCUMENT>(.*?)</DOCUMENT>',full,re.I|re.S)];texts=[]
 for doc in docs:
  if not re.search(r'<TYPE>\s*EX-99\.[12](?:\s|<)',doc,re.I):continue
  m=re.search(r'<TEXT>(.*?)</TEXT>',doc,re.I|re.S)
  if m:texts.append(' '.join(BeautifulSoup(m.group(1),'html.parser').stripped_strings))
 txt=re.sub(r'\s+',' ',' '.join(texts));hits={k:bool(re.search(v,txt)) for k,v in patterns.items()}
 controls.append({'accession':item['accession'],'filing_date':item['filing_date'],'raw_sha256':item['raw_sha256'],'EX99_document_text_count':len(texts),'source_phrases':hits,'source_text_sha256':hashlib.sha256(txt.encode()).hexdigest()})
assert controls[0]['source_phrases']['pre_Q1_outlook_includes_next_quarter'] and controls[1]['source_phrases']['Q2_closed_April_27'] and controls[1]['source_phrases']['Q2_Data_Center_includes_Mellanox'],controls
mask=[]
for item in plan['exact_definition_transition_accessions']:
 e=events[item['accession']]
 assert all(e[k]==item[k] for k in item) and e['status']=='numeric_same_release_table'
 mask.append({'accession':e['accession'],'filing_date':e['filing_date'],'first_conservative_Qlib_signal_date':e['first_conservative_Qlib_signal_date'],'reported_same_release_YoY':e['same_release_YoY'],'comparability':'unknown_current_includes_Mellanox_prior_year_excludes'})
post=events['0001045810-21-000128'];assert post['status']=='numeric_same_release_table' and post['filing_date']=='2021-08-18'
result={'decision':'four_original_NVDA_Data_Center_YoY_events_Mellanox_incomparable_source_mask','plan_sha256':sha(card/'plan.json'),'controls':controls,'masked_events':mask,'first_post_boundary_comparable_reference':{'accession':post['accession'],'filing_date':post['filing_date'],'first_conservative_Qlib_signal_date':post['first_conservative_Qlib_signal_date']},'raw_original_reads':2,'numeric_values_changed':False,'other_definition_changes_qualified_absent':False,'issuer_first_publication_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
settings=_settings(root);files=[card/'plan.json',card/'result.json']
with tempfile.TemporaryDirectory(prefix='nvda-mellanox-') as tmp:
 a=Path(tmp)/'source.tar.gz'
 with tarfile.open(a,'w:gz') as out:
  for p in files:out.add(p,arcname=str(p.relative_to(root)))
 digest=hashlib.sha256(a.read_bytes()).hexdigest();response=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=a,executor_id='soxl-minute-research');rec=response['artifact'];assert rec['sha256']==digest and rec['size_bytes']==a.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest;restore=Path(tmp)/'restore.tar.gz'
 with restore.open('wb') as out:proc=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert proc.returncode==0 and hashlib.sha256(restore.read_bytes()).hexdigest()==digest
 with tarfile.open(restore,'r:gz') as z:
  names={m.name:m for m in z.getmembers() if m.isfile()};assert set(names)=={str(p.relative_to(root)) for p in files}
  for p in files:assert hashlib.sha256(z.extractfile(names[str(p.relative_to(root))]).read()).hexdigest()==sha(p)
receipt={'status':'source_semantics_NAS_scoped_restore_verified','sha256':digest,'size_bytes':rec['size_bytes'],'nas_path':remote,'members':{str(p.relative_to(root)):sha(p) for p in files},'verified':True};(card/'nas_source_archive.json').write_text(json.dumps(receipt,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_archive':'nas_source_archive.json','raw_original_reads':2,'model_fits':0,'native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Four source events require a structural comparability unknown mask; other platform changes are not certified absent.','Original SEC filing next-session proxy does not prove first issuer publication.','No SOXX outcome, Qlib factor fit or account.']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(json.dumps({'masked_accessions':[x['accession'] for x in mask],'controls':controls,'nas_sha256':digest},indent=2))


# Mellanox 定义来源对照：第一版失败

冻结的两份原始 SEC 对照中，2020-05-21 一份确认下一季度指引将纳入 Mellanox；第二份误选为 2021-08-18，无法认证 2020 年首次纳入季度。严格来源断言在两次原件读取后失败，0 HTTP、0 市场收益、0 数值因子、0 拟合、0 账户。原计划和失败观测保留，不更换本版对照或重跑。下一版只读 2020-08-19 原始附件，并与已保存原件身份相连。
